# 27. 프로덕션 Pickle 내보내기

## 개요

학습된 모델을 SelF 프로덕션 환경에 배포하기 위한 최종 Pickle 파일 생성.

**Pickle 구조 (v2.0)**:
```python
{
    'version': '2.0.0',
    'algorithm': 'ALS',
    'metadata': {...},
    'components': {
        'user_embeddings': bytes,
        'product_embeddings': bytes,
        'user_id_to_idx': dict,
        'idx_to_product_id': dict,
        'global_popular_products': list,
        'category_popular_products': dict,
    },
    'hyperparameters': {...},
    'metrics': {...},
}
```

**프로덕션 최적화**:
- ALS 32D (GitHub 100MB 제한 준수)
- 파일 크기: ~30MB

## 1. 환경 설정

In [1]:
import os
import sys
import json
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))

from utils.personalization import (
    OptimizedALSRecommender,
    create_optimized_pickle,
    load_optimized_pickle,
)

DATA_PATH = PROJECT_ROOT / 'data' / 'instacart'
# 폴더 구조 수정: data/models/ → data/processed/personalization/
MODEL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'personalization'
PROD_PATH = PROJECT_ROOT / 'pred' / 'models'

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"모델 경로: {MODEL_PATH}")
print(f"프로덕션 경로: {PROD_PATH}")

프로젝트 루트: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone
모델 경로: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\data\processed\personalization
프로덕션 경로: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\pred\models


## 2. 모델 및 메타데이터 로드

In [2]:
# ALS 32D 모델 로드 (GitHub 100MB 제한 준수)
als_model = OptimizedALSRecommender.load(str(MODEL_PATH / 'als_32dim_model.pkl'))

print(f"ALS 모델 로드 완료")
print(f"  - factors: {als_model.factors}")
print(f"  - User Factors: {als_model.user_factors.shape}")
print(f"  - Item Factors: {als_model.item_factors.shape}")

INFO:utils.personalization.als_recommender:[로드 완료] d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\data\processed\personalization\als_32dim_model.pkl
INFO:utils.personalization.als_recommender:  • 버전: 2.0.0
INFO:utils.personalization.als_recommender:  • 차원: 32
INFO:utils.personalization.als_recommender:  • 유저: 206,209
INFO:utils.personalization.als_recommender:  • 아이템: 49,685


ALS 모델 로드 완료
  - factors: 32
  - User Factors: (206209, 32)
  - Item Factors: (49685, 32)


In [3]:
# 평가 결과 로드
with open(MODEL_PATH / 'evaluation_results.json', 'r') as f:
    eval_results = json.load(f)

best_model = eval_results['best_model']
print(f"최고 성능 모델: {best_model}")

최고 성능 모델: ALS-32D


In [4]:
# 피처 데이터 로드
with open(DATA_PATH / 'features.json', 'r', encoding='utf-8') as f:
    features = json.load(f)

print(f"피처 데이터 로드 완료")

피처 데이터 로드 완료


## 3. 프로덕션 Pickle 생성

In [5]:
# 프로덕션 Pickle 구조 정의
def create_production_pickle(als_model, features, eval_results):
    """
    SelF 프로덕션용 Pickle 생성
    
    구조:
    - version: Pickle 버전
    - algorithm: 알고리즘 이름
    - metadata: 모델 메타데이터
    - components: 핵심 컴포넌트
    - hyperparameters: 하이퍼파라미터
    - metrics: 평가 지표
    """
    
    # 임베딩을 bytes로 변환 (메모리 최적화)
    user_embeddings_bytes = als_model.user_factors.astype(np.float32).tobytes()
    product_embeddings_bytes = als_model.item_factors.astype(np.float32).tobytes()
    
    pickle_data = {
        'version': '2.0.0',
        'algorithm': 'ALS',
        'created_at': datetime.now().isoformat(),
        
        'metadata': {
            'n_users': als_model.user_factors.shape[0],
            'n_items': als_model.item_factors.shape[0],
            'factors': als_model.factors,
            'dtype': 'float32',
        },
        
        'components': {
            'user_embeddings': {
                'data': user_embeddings_bytes,
                'shape': als_model.user_factors.shape,
            },
            'product_embeddings': {
                'data': product_embeddings_bytes,
                'shape': als_model.item_factors.shape,
            },
            'user_id_to_idx': als_model.user_id_to_idx,
            'idx_to_user_id': als_model.idx_to_user_id,
            'product_id_to_idx': als_model.item_id_to_idx,
            'idx_to_product_id': als_model.idx_to_item_id,
            'global_popular_products': als_model.global_popular_products,
            'category_popular_products': getattr(als_model, 'category_popular_products', {}),
        },
        
        'hyperparameters': {
            'factors': als_model.factors,
            'regularization': als_model.regularization,
            'alpha': als_model.alpha,
            'cbf_weight': 0.4,  # 최적화된 가중치
            'cf_weight': 0.6,
            'filter_already_liked_items': False,  # 식료품 특화
        },
        
        'metrics': eval_results['models'].get('ALS-128D', {}).get('metrics', {}),
    }
    
    return pickle_data

print("프로덕션 Pickle 생성 함수 정의 완료")

프로덕션 Pickle 생성 함수 정의 완료


In [6]:
%%time

# Pickle 생성
prod_pickle = create_production_pickle(als_model, features, eval_results)

print(f"Pickle 생성 완료")
print(f"  - 버전: {prod_pickle['version']}")
print(f"  - 알고리즘: {prod_pickle['algorithm']}")
print(f"  - 사용자 수: {prod_pickle['metadata']['n_users']:,}")
print(f"  - 상품 수: {prod_pickle['metadata']['n_items']:,}")

Pickle 생성 완료
  - 버전: 2.0.0
  - 알고리즘: ALS
  - 사용자 수: 206,209
  - 상품 수: 49,685
CPU times: total: 15.6 ms
Wall time: 9.67 ms


## 4. Pickle 저장

In [7]:
# 프로덕션 경로에 저장
output_path = PROD_PATH / 'self_personalized_v2.pkl'

with open(output_path, 'wb') as f:
    pickle.dump(prod_pickle, f, protocol=pickle.HIGHEST_PROTOCOL)

file_size = output_path.stat().st_size / 1024**2

print(f"\n프로덕션 Pickle 저장 완료")
print(f"  - 경로: {output_path}")
print(f"  - 파일 크기: {file_size:.2f} MB")


프로덕션 Pickle 저장 완료
  - 경로: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\pred\models\self_personalized_v2.pkl
  - 파일 크기: 35.24 MB


In [8]:
# 모델 경로에도 백업 저장
backup_path = MODEL_PATH / 'self_personalized_v2.pkl'

with open(backup_path, 'wb') as f:
    pickle.dump(prod_pickle, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"백업 저장 완료: {backup_path}")

백업 저장 완료: d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\data\processed\personalization\self_personalized_v2.pkl


## 5. Pickle 검증

In [9]:
# 로드 테스트
with open(output_path, 'rb') as f:
    loaded_pickle = pickle.load(f)

print("Pickle 로드 테스트:")
print(f"  - 버전: {loaded_pickle['version']}")
print(f"  - 알고리즘: {loaded_pickle['algorithm']}")
print(f"  - 컴포넌트: {list(loaded_pickle['components'].keys())}")

Pickle 로드 테스트:
  - 버전: 2.0.0
  - 알고리즘: ALS
  - 컴포넌트: ['user_embeddings', 'product_embeddings', 'user_id_to_idx', 'idx_to_user_id', 'product_id_to_idx', 'idx_to_product_id', 'global_popular_products', 'category_popular_products']


In [10]:
# 임베딩 복원 테스트
user_emb_data = loaded_pickle['components']['user_embeddings']
user_embeddings = np.frombuffer(
    user_emb_data['data'], 
    dtype=np.float32
).reshape(user_emb_data['shape'])

product_emb_data = loaded_pickle['components']['product_embeddings']
product_embeddings = np.frombuffer(
    product_emb_data['data'], 
    dtype=np.float32
).reshape(product_emb_data['shape'])

print("임베딩 복원 테스트:")
print(f"  - User Embeddings: {user_embeddings.shape}")
print(f"  - Product Embeddings: {product_embeddings.shape}")

# 원본과 비교
assert np.allclose(user_embeddings, als_model.user_factors), "User embeddings 불일치!"
assert np.allclose(product_embeddings, als_model.item_factors), "Item embeddings 불일치!"

print("\n✓ 임베딩 무결성 검증 완료")

임베딩 복원 테스트:
  - User Embeddings: (206209, 32)
  - Product Embeddings: (49685, 32)

✓ 임베딩 무결성 검증 완료


In [11]:
# 추천 테스트
def _get_mapping_value(mapping, key):
    """키 타입이 달라도 매핑 값을 최대한 찾아 반환한다."""
    
    candidates = [key]
    if isinstance(key, str):
        candidates.append(key.strip())
        if key.isdigit():
            try:
                candidates.append(int(key))
            except ValueError:
                pass
    else:
        candidates.append(str(key))
        try:
            candidates.append(int(key))
        except (TypeError, ValueError):
            pass
    
    for candidate in candidates:
        if candidate in mapping:
            return mapping[candidate]
    return None

def recommend_from_pickle(pickle_data, user_id, n_items=10):
    """Pickle에서 직접 추천을 생성한다(프로덕션 시뮬레이션)."""
    
    # 임베딩 복원
    user_emb_data = pickle_data['components']['user_embeddings']
    user_factors = np.frombuffer(
        user_emb_data['data'], dtype=np.float32
    ).reshape(user_emb_data['shape'])
    
    item_emb_data = pickle_data['components']['product_embeddings']
    item_factors = np.frombuffer(
        item_emb_data['data'], dtype=np.float32
    ).reshape(item_emb_data['shape'])
    
    # 매핑 조회
    user_id_to_idx = pickle_data['components']['user_id_to_idx']
    idx_to_product_id = pickle_data['components']['idx_to_product_id']
    
    # 사용자 인덱스 조회
    user_idx = _get_mapping_value(user_id_to_idx, user_id)
    if user_idx is None:
        # 콜드 스타트: 인기 상품 반환
        popular_products = pickle_data['components']['global_popular_products'][:n_items]
        return [(pid, 0.0) for pid in popular_products]
    
    # 점수 계산
    user_vector = user_factors[user_idx]
    scores = item_factors @ user_vector
    
    # 상위 N개
    top_indices = np.argsort(scores)[::-1][:n_items]
    
    recommendations = []
    for idx in top_indices:
        product_id = _get_mapping_value(idx_to_product_id, idx)
        if product_id is None:
            product_id = int(idx)
        recommendations.append((product_id, float(scores[idx])))
    return recommendations

# 테스트
test_user_id = next(iter(loaded_pickle['components']['user_id_to_idx'].keys()))
pickle_recs = recommend_from_pickle(loaded_pickle, test_user_id, n_items=5)

print(f"\nPickle 기반 추천 테스트 (user_id={test_user_id}):")
for pid, score in pickle_recs:
    print(f"  [{score:.4f}] product_id={pid}")


Pickle 기반 추천 테스트 (user_id=1):
  [0.9690] product_id=13176
  [0.9662] product_id=12341
  [0.9309] product_id=6184
  [0.9096] product_id=196
  [0.9078] product_id=16797


## 6. 메모리 효율성 분석

In [12]:
# 메모리 사용량 분석
print("메모리 효율성 분석:")
print("="*50)

# 임베딩 크기
user_mem = als_model.user_factors.nbytes / 1024**2
item_mem = als_model.item_factors.nbytes / 1024**2

print(f"\n1. 임베딩 메모리")
print(f"   - User ({als_model.user_factors.shape}): {user_mem:.2f} MB")
print(f"   - Item ({als_model.item_factors.shape}): {item_mem:.2f} MB")
print(f"   - 합계: {user_mem + item_mem:.2f} MB")

# SVD 128D와 비교
svd_128d_user_mem = als_model.user_factors.shape[0] * 128 * 4 / 1024**2
svd_128d_item_mem = als_model.item_factors.shape[0] * 128 * 4 / 1024**2

print(f"\n2. SVD 128D 대비 비교 (추정)")
print(f"   - SVD 128D User: {svd_128d_user_mem:.2f} MB")
print(f"   - SVD 128D Item: {svd_128d_item_mem:.2f} MB")
print(f"   - SVD 128D 합계: {svd_128d_user_mem + svd_128d_item_mem:.2f} MB")

reduction = (1 - (user_mem + item_mem) / (svd_128d_user_mem + svd_128d_item_mem)) * 100
print(f"   - 절감률: {reduction:.1f}%")

# Pickle 파일 크기
print(f"\n3. Pickle 파일 크기")
print(f"   - self_personalized_v2.pkl: {file_size:.2f} MB")

메모리 효율성 분석:

1. 임베딩 메모리
   - User ((206209, 32)): 25.17 MB
   - Item ((49685, 32)): 6.07 MB
   - 합계: 31.24 MB

2. SVD 128D 대비 비교 (추정)
   - SVD 128D User: 100.69 MB
   - SVD 128D Item: 24.26 MB
   - SVD 128D 합계: 124.95 MB
   - 절감률: 75.0%

3. Pickle 파일 크기
   - self_personalized_v2.pkl: 35.24 MB


## 7. 프로덕션 메타데이터 업데이트

In [13]:
# model_metadata.json 업데이트 정보
metadata_update = {
    'self_personalized': {
        'active_version': 'v2',
        'versions': {
            'v2': {
                'file': 'self_personalized_v2.pkl',
                'algorithm': 'ALS',
                'factors': 32,
                'created_at': datetime.now().isoformat(),
                'metrics': {
                    'Recall@10': eval_results['models'].get('ALS-32D', {}).get('metrics', {}).get('Recall@10', 0),
                    'NDCG@10': eval_results['models'].get('ALS-32D', {}).get('metrics', {}).get('NDCG@10', 0),
                },
                'memory_mb': file_size,
                'github_compatible': file_size < 100,  # GitHub 100MB 제한 체크
            },
        },
    },
}

print("model_metadata.json 업데이트 정보:")
print(json.dumps(metadata_update, indent=2))

model_metadata.json 업데이트 정보:
{
  "self_personalized": {
    "active_version": "v2",
    "versions": {
      "v2": {
        "file": "self_personalized_v2.pkl",
        "algorithm": "ALS",
        "factors": 32,
        "created_at": "2025-12-22T00:39:17.321315",
        "metrics": {
          "Recall@10": 0.10508837978963737,
          "NDCG@10": 0.11518543007576482
        },
        "memory_mb": 35.23950481414795,
        "github_compatible": true
      }
    }
  }
}


## 8. 요약

In [15]:
summary = f"""
==========================================
프로덕션 Pickle 내보내기 완료
==========================================

1. 생성된 파일
   - {output_path}
   - 파일 크기: {file_size:.2f} MB

2. Pickle 구조
   - 버전: 2.0.0
   - 알고리즘: ALS 32차원
   - 사용자 수: {prod_pickle['metadata']['n_users']:,}
   - 상품 수: {prod_pickle['metadata']['n_items']:,}

3. 주요 컴포넌트
   - user_embeddings: ({prod_pickle['metadata']['n_users']}, 32)
   - product_embeddings: ({prod_pickle['metadata']['n_items']}, 32)
   - global_popular_products: {len(prod_pickle['components']['global_popular_products'])}개
   - category_popular_products: {len(prod_pickle['components']['category_popular_products'])}개 카테고리

4. 프로덕션 배포
   - pred/models/self_personalized_v2.pkl
   - pred/models/model_metadata.json 업데이트 필요

==========================================
"""

print(summary)


프로덕션 Pickle 내보내기 완료

1. 생성된 파일
   - d:\VSC_Project\SSAFY\SSAFY_Class_18_Team_4_Final_Capstone\SSAFY_Class_18_Team_4_Final_Capstone\pred\models\self_personalized_v2.pkl
   - 파일 크기: 35.24 MB

2. Pickle 구조
   - 버전: 2.0.0
   - 알고리즘: ALS 32차원
   - 사용자 수: 206,209
   - 상품 수: 49,685

3. 주요 컴포넌트
   - user_embeddings: (206209, 32)
   - product_embeddings: (49685, 32)
   - global_popular_products: 200개
   - category_popular_products: 0개 카테고리

4. 프로덕션 배포
   - pred/models/self_personalized_v2.pkl
   - pred/models/model_metadata.json 업데이트 필요




## 다음 단계

1. **pred/ml/models/self_personalized.py** 수정
   - v2 Pickle 로드 로직 추가
   - 하이브리드 추천 로직 통합

2. **pred/models/model_metadata.json** 업데이트
   - active_version: v2 설정

3. **테스트 및 배포**
   - API 통합 테스트
   - 성능 모니터링